# Notebook 03 — Modélisation Nutrition (Ensemble RF + MLP)

**Objectif** : Développer, entraîner et évaluer le modèle Ensemble
(Random Forest + MLP Neural Network) de prédiction du risque malnutrition aiguë.

**Pipeline** :
1. Construction dataset (32 features, incluant feature croisée paludisme)
2. Analyse EDA nutritionnelle (GAM, FCS, soudure, prix)
3. Entraînement composants RF et MLP
4. Calibration de l'ensemble
5. Évaluation classification + régression GAM
6. Explicabilité SHAP (ensemble)
7. Analyse saisonnière et régionale
8. Test feature croisée paludisme→nutrition
9. Sélection de recettes (RecipeSelector)
10. Validation seuils OMS et UNICEF

---

In [1]:
# ── Imports ──────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path().resolve().parents[1]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import date, timedelta

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})

UNICEF_BLUE   = '#00AEEF'
UNICEF_DARK   = '#374EA2'
UNICEF_RED    = '#E2231A'
UNICEF_GREEN  = '#80BD41'
UNICEF_ORANGE = '#F26A21'
UNICEF_YELLOW = '#FFCC00'

# Couleurs seuils OMS
COLOR_ACCEPTABLE = '#4CAF50'
COLOR_ALERTE     = '#FFC107'
COLOR_URGENCE    = '#FF9800'
COLOR_CRISE      = '#F44336'

print('Environnement prêt')
print(f'   numpy {np.__version__} | pandas {pd.__version__}')

Environnement prêt
   numpy 1.26.4 | pandas 2.2.2


## 1. Construction et Analyse du Dataset


In [ ]:
# ── Génération dataset synthétique nutrition ────────────────────
from ml.training_scripts.train_nutrition import _generer_donnees_synthetiques_nutrition
from src.models.nutrition_predictor import NUTRITION_FEATURE_NAMES

print('Construction dataset nutrition...')
X_raw, y_raw, y_gam_raw, feature_names = _generer_donnees_synthetiques_nutrition()

df = pd.DataFrame(X_raw, columns=feature_names)
df['score_risque'] = y_raw
df['gam_pct']      = y_gam_raw
df['classe_gam']   = pd.cut(
    y_gam_raw,
    bins=[0, 5, 10, 15, 100],
    labels=['Acceptable (<5%)', 'Alerte (5-10%)', 'Urgence (10-15%)', 'Crise (≥15%)']
)

print(f'Dataset : {X_raw.shape[0]} samples × {X_raw.shape[1]} features')
print(f'   GAM moyen   : {y_gam_raw.mean():.2f}% ± {y_gam_raw.std():.2f}%')
print(f'   GAM médiane : {np.median(y_gam_raw):.2f}%')
print(f'   Score risque moyen : {y_raw.mean():.3f}')
print(f'\n32 Features :')
categories = [
    ('Historique nutritionnel', feature_names[:5]),
    ('Sécurité alimentaire', feature_names[5:8]),
    ('Prix alimentaires', feature_names[8:11]),
    ('Disponibilité', feature_names[11:16]),
    ('Feature croisée paludisme', feature_names[16:17]),
    ('Climatique', feature_names[17:20]),
    ('Socio-économique', feature_names[20:23]),
    ('Temporel', feature_names[23:26]),
    ('Géographique + Démo', feature_names[26:]),
]
for cat, feats in categories:
    print(f'  [{cat}]')
    for f in feats:
        print(f'    - {f}')

In [ ]:
# ── Distribution GAM et répartition OMS ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Histogramme GAM
ax1 = axes[0]
n, bins, patches = ax1.hist(y_gam_raw, bins=40, edgecolor='white', alpha=0.0)
for patch, left_edge in zip(patches, bins):
    if left_edge < 5:   patch.set_facecolor(COLOR_ACCEPTABLE)
    elif left_edge < 10: patch.set_facecolor(COLOR_ALERTE)
    elif left_edge < 15: patch.set_facecolor(COLOR_URGENCE)
    else:                patch.set_facecolor(COLOR_CRISE)
    patch.set_alpha(0.85)

for seuil, couleur, label in [
    (5,  COLOR_ALERTE,  'Seuil alerte OMS (5%)'),
    (10, COLOR_URGENCE, 'Seuil urgence OMS (10%)'),
    (15, COLOR_CRISE,   'Seuil crise OMS (15%)'),
]:
    ax1.axvline(x=seuil, color=couleur, linestyle='--', linewidth=2.5, label=label)
ax1.axvline(x=y_gam_raw.mean(), color='black', linestyle='-', linewidth=2,
            label=f'Moyenne ({y_gam_raw.mean():.1f}%)')
ax1.set_title('Distribution du Taux GAM\n(Global Acute Malnutrition)', fontweight='bold')
ax1.set_xlabel('GAM (%)')
ax1.set_ylabel('Fréquence')
ax1.legend(fontsize=8)

# Répartition classes OMS
ax2 = axes[1]
counts = df['classe_gam'].value_counts()
colors_oms = [COLOR_ACCEPTABLE, COLOR_ALERTE, COLOR_URGENCE, COLOR_CRISE]
wedges, texts, autotexts = ax2.pie(
    counts.values,
    labels=counts.index,
    colors=colors_oms,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=2),
)
for text in autotexts:
    text.set_fontsize(9)
    text.set_fontweight('bold')
ax2.set_title('Répartition selon Seuils OMS', fontweight='bold')

# Boxplot GAM par score de risque binned
ax3 = axes[2]
score_bins = pd.cut(y_raw, bins=[0, 0.25, 0.5, 0.75, 1.0],
                    labels=['Faible', 'Moyen', 'Élevé', 'Très élevé'])
df_box = pd.DataFrame({'score_bin': score_bins, 'gam_pct': y_gam_raw})
groups = [df_box[df_box['score_bin']==s]['gam_pct'].values
          for s in ['Faible', 'Moyen', 'Élevé', 'Très élevé']]
bplot = ax3.boxplot(groups, patch_artist=True,
                    boxprops=dict(alpha=0.8),
                    medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bplot['boxes'],
                         [COLOR_ACCEPTABLE, COLOR_ALERTE, COLOR_URGENCE, COLOR_CRISE]):
    patch.set_facecolor(color)
ax3.set_xticklabels(['Faible', 'Moyen', 'Élevé', 'Très élevé'])
ax3.set_ylabel('GAM (%)')
ax3.set_title('GAM% par Niveau de Score Risque', fontweight='bold')
for seuil, couleur in [(5, COLOR_ALERTE), (10, COLOR_URGENCE), (15, COLOR_CRISE)]:
    ax3.axhline(y=seuil, color=couleur, linestyle=':', alpha=0.7, linewidth=1.5)

plt.suptitle('Analyse de la Variable Cible — Taux GAM Madagascar',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. EDA — Corrélations et Features Clés Nutrition


In [ ]:
# ── Analyse de corrélation features nutrition vs GAM ────────────
corr_gam = df[list(feature_names) + ['gam_pct']].corr()['gam_pct'].drop('gam_pct')
corr_gam = corr_gam.sort_values()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# Corrélation avec GAM
ax1 = axes[0]
colors_corr = [COLOR_CRISE if v > 0.1 else
               COLOR_URGENCE if v > 0.05 else
               COLOR_ACCEPTABLE if v < -0.1 else
               UNICEF_BLUE if v < 0 else '#CCC'
               for v in corr_gam.values]
bars = ax1.barh(corr_gam.index, corr_gam.values,
                color=colors_corr, alpha=0.85, edgecolor='white')
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.axvline(x=0.2, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax1.axvline(x=-0.2, color='blue', linestyle='--', alpha=0.4, linewidth=1)
ax1.set_title('Corrélation Pearson — Features vs Taux GAM%', fontweight='bold')
ax1.set_xlabel('Coefficient de corrélation avec GAM%')

# Annotations top features
for feat, val in corr_gam.nlargest(5).items():
    ax1.text(val + 0.005, list(corr_gam.index).index(feat),
             f'+{val:.3f} ↑GAM', va='center', fontsize=7.5,
             color=COLOR_CRISE, fontweight='bold')
for feat, val in corr_gam.nsmallest(5).items():
    ax1.text(val - 0.005, list(corr_gam.index).index(feat),
             f'{val:.3f} ↓GAM', va='center', fontsize=7.5,
             color=COLOR_ACCEPTABLE, fontweight='bold', ha='right')

# Scatter matrix features clés vs GAM
ax2 = axes[1]
top_neg = corr_gam.nsmallest(1).index[0]  # Feature protectrice
top_pos = corr_gam.nlargest(1).index[0]   # Feature de risque

sc1 = ax2.scatter(df[top_pos], df['gam_pct'],
                  c=y_raw, cmap='RdYlGn_r', s=15, alpha=0.5)
plt.colorbar(sc1, ax=ax2, label='Score risque nutrition')
# Ligne de tendance
z = np.polyfit(df[top_pos], df['gam_pct'], 1)
p = np.poly1d(z)
x_line = np.linspace(df[top_pos].min(), df[top_pos].max(), 100)
ax2.plot(x_line, p(x_line), UNICEF_RED, linewidth=2.5,
         label=f'Tendance (r={corr_gam[top_pos]:.3f})')
for seuil, couleur in [(5, COLOR_ALERTE), (10, COLOR_URGENCE), (15, COLOR_CRISE)]:
    ax2.axhline(y=seuil, color=couleur, linestyle='--', alpha=0.7, linewidth=1.5)
ax2.set_xlabel(top_pos, fontsize=10)
ax2.set_ylabel('GAM%', fontsize=10)
ax2.set_title(f'Feature principale vs GAM%\n({top_pos})', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Analyse de Corrélation — Features Nutrition vs GAM%',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Features les plus corrélées avec GAM% :')
print('\n  ↑ Augmentent le risque (corrélation positive) :')
for f, v in corr_gam.nlargest(5).items():
    print(f'    {f:35s}: +{v:.4f}')
print('\n  ↓ Diminuent le risque (corrélation négative) :')
for f, v in corr_gam.nsmallest(5).items():
    print(f'    {f:35s}: {v:.4f}')

## 3. Impact de la Feature Croisée Paludisme → Nutrition


In [ ]:
# ── Analyse feature croisée : score paludisme dans nutrition ─────
# Dans NUTRITION_FEATURE_NAMES, 'score_paludisme' est à l'index 16
idx_pal = list(feature_names).index('score_paludisme')
df['score_paludisme_bin'] = pd.cut(
    df['score_paludisme'], bins=[0, 0.25, 0.5, 0.75, 1],
    labels=['Faible (<0.25)', 'Moyen (0.25-0.5)', 'Élevé (0.5-0.75)', 'Très élevé (>0.75)']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GAM moyen par niveau de risque paludisme
ax1 = axes[0]
gam_par_pal = df.groupby('score_paludisme_bin', observed=True)['gam_pct'].agg(['mean','std','count'])
colors_pal = [COLOR_ACCEPTABLE, COLOR_ALERTE, COLOR_URGENCE, COLOR_CRISE]
bars = ax1.bar(
    range(len(gam_par_pal)), gam_par_pal['mean'],
    color=colors_pal, edgecolor='white', alpha=0.85,
    yerr=gam_par_pal['std'], capsize=5,
    error_kw=dict(elinewidth=1.5, ecolor='black', capthick=1.5)
)
for i, (bar, row) in enumerate(zip(bars, gam_par_pal.itertuples())):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + row.std + 0.2,
             f'{row.mean:.1f}%\n(n={row.count})',
             ha='center', fontsize=8.5, fontweight='bold')
ax1.set_xticks(range(len(gam_par_pal)))
ax1.set_xticklabels(gam_par_pal.index, rotation=10, fontsize=9)
ax1.set_ylabel('GAM moyen (%)')
ax1.set_title('Impact du Score Paludisme sur GAM%\n(Feature croisée — co-morbidité)',
               fontweight='bold')
for seuil, col in [(5, COLOR_ALERTE), (10, COLOR_URGENCE), (15, COLOR_CRISE)]:
    ax1.axhline(y=seuil, color=col, linestyle=':', alpha=0.7, linewidth=1.5)

# Scatter paludisme vs GAM coloré par score_fcs
ax2 = axes[1]
idx_fcs = list(feature_names).index('score_fcs')
sc = ax2.scatter(
    df['score_paludisme'], df['gam_pct'],
    c=df['score_fcs'], cmap='RdYlGn',
    s=12, alpha=0.4,
    vmin=df['score_fcs'].quantile(0.05),
    vmax=df['score_fcs'].quantile(0.95)
)
cbar = plt.colorbar(sc, ax=ax2)
cbar.set_label('Score FCS (vert=bon)', fontsize=9)
z = np.polyfit(df['score_paludisme'], df['gam_pct'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 1, 100)
ax2.plot(x_line, p(x_line), UNICEF_RED, linewidth=2.5, linestyle='--',
         label=f'Tendance (r={corr_gam["score_paludisme"]:.3f})')
for seuil, col in [(5, COLOR_ALERTE), (10, COLOR_URGENCE), (15, COLOR_CRISE)]:
    ax2.axhline(y=seuil, color=col, linestyle=':', alpha=0.7)
ax2.set_xlabel('Score de risque paludisme')
ax2.set_ylabel('GAM%')
ax2.set_title('Paludisme vs Malnutrition\n(coloré par FCS)', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle(
    'Feature Croisée Paludisme → Nutrition\n'
    'Co-morbidité : paludisme chronique → malnutrition aiguë',
    fontsize=12, fontweight='bold', color=UNICEF_DARK
)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_comorbidite.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistique
gam_faible_pal = df[df['score_paludisme_bin']=='Faible (<0.25)']['gam_pct'].mean()
gam_fort_pal   = df[df['score_paludisme_bin']=='Très élevé (>0.75)']['gam_pct'].mean()
print(f'\nImpact co-morbidité paludisme → nutrition :')
print(f'   GAM moyen (risque pal. faible) : {gam_faible_pal:.1f}%')
print(f'   GAM moyen (risque pal. élevé)  : {gam_fort_pal:.1f}%')
print(f'   Ratio (aggravation)            : {gam_fort_pal/max(gam_faible_pal,0.1):.2f}×')

## 4. Prétraitement et Entraînement Ensemble


In [ ]:
# ── Split et scaling ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ml.training_scripts.train_nutrition import _train_test_split_nutrition

X_tr, X_te, y_tr, y_te, yg_tr, yg_te = _train_test_split_nutrition(
    X_raw, y_raw, y_gam_raw, test_size=0.20
)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

print(f'Train : {X_tr.shape} | Test : {X_te.shape}')
print(f'GAM train : {yg_tr.mean():.2f}% ± {yg_tr.std():.2f}%')
print(f'GAM test  : {yg_te.mean():.2f}% ± {yg_te.std():.2f}%')

In [ ]:
# ── Entraînement Ensemble RF + MLP ──────────────────────────────
from src.models.nutrition_predictor import NutritionPredictor

print('🏋️  Entraînement NutritionPredictor (RF + MLP)...')
model = NutritionPredictor()
model._build_model()

model.fit(
    X_tr_sc, y_tr,
    feature_names=list(feature_names),
    y_gam=yg_tr,
)
model._scaler = scaler

# Évaluation initiale
metriques = model.evaluate(X_te_sc, y_te, y_gam_test=yg_te)
print(f'\nMétriques NutritionPredictor :')
for k, v in metriques.items():
    if isinstance(v, (int, float)):
        print(f'   {k:25s}: {v:.4f}')

## 5. Analyse des Composants RF et MLP Séparément


In [ ]:
# ── Performance RF vs MLP vs Ensemble ───────────────────────────
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from sklearn.metrics import roc_curve, precision_recall_curve

y_clf_test = (y_te >= 0.25).astype(int)

# Prédictions de chaque composant
rf_proba  = model._rf.predict_proba(X_te_sc)[:, 1]
mlp_proba = model._mlp.predict_proba(X_te_sc)[:, 1]
ens_proba = model._predict_raw(X_te_sc)

composants = {
    f'Random Forest (poids 60%)':     rf_proba,
    f'MLP Neural Network (poids 40%)': mlp_proba,
    f'Ensemble RF+MLP':                ens_proba,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_comp = [UNICEF_BLUE, UNICEF_GREEN, UNICEF_RED]
styles_comp = ['--', ':', '-']
lw_comp     = [1.5, 1.5, 2.5]

metrics_table = []
for (nom, proba), color, ls, lw in zip(
    composants.items(), colors_comp, styles_comp, lw_comp
):
    auc  = roc_auc_score(y_clf_test, proba)
    ap   = average_precision_score(y_clf_test, proba)
    f1   = f1_score(y_clf_test, (proba>=0.5).astype(int), zero_division=0)
    metrics_table.append({'Composant': nom, 'AUC-ROC': auc, 'Avg Precision': ap, 'F1': f1})

    fpr, tpr, _ = roc_curve(y_clf_test, proba)
    axes[0].plot(fpr, tpr, color=color, linewidth=lw, linestyle=ls,
                 label=f'{nom} (AUC={auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_clf_test, proba)
    axes[1].plot(rec, prec, color=color, linewidth=lw, linestyle=ls,
                 label=f'{nom} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1], 'k--', alpha=0.3, label='Aléatoire')
axes[0].set_title('Courbes ROC — RF vs MLP vs Ensemble', fontweight='bold')
axes[0].set_xlabel('Taux Faux Positifs')
axes[0].set_ylabel('Taux Vrais Positifs')
axes[0].legend(fontsize=9)
axes[0].axhline(y=0.68, color='gray', linestyle=':', alpha=0.5)
axes[0].text(0.02, 0.70, 'Seuil UNICEF recall=0.68', fontsize=8, color='gray')

axes[1].axhline(y=y_clf_test.mean(), color='gray', linestyle='--', alpha=0.5,
                label=f'Baseline ({y_clf_test.mean():.2f})')
axes[1].set_title('Courbes Précision-Rappel', fontweight='bold')
axes[1].set_xlabel('Rappel')
axes[1].set_ylabel('Précision')
axes[1].legend(fontsize=9)

plt.suptitle('Comparaison des Composants — Ensemble RF + MLP',
             fontsize=12, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_roc_composants.png', dpi=150, bbox_inches='tight')
plt.show()

df_metrics = pd.DataFrame(metrics_table).set_index('Composant')
print('\nTableau comparatif des composants :')
display(df_metrics.round(4))

best = df_metrics['AUC-ROC'].idxmax()
print(f'\nMeilleur composant AUC : {best}')
print(f'   Gain Ensemble vs RF seul  : +{(df_metrics.loc["Ensemble RF+MLP","AUC-ROC"]-df_metrics.loc["Random Forest (poids 60%)","AUC-ROC"]):.4f}')

## 6. Évaluation du Régresseur GAM


In [ ]:
# ── Performance du régresseur GAM ───────────────────────────────
y_gam_pred = model._reg.predict(X_te_sc)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae  = mean_absolute_error(yg_te, y_gam_pred)
rmse = np.sqrt(mean_squared_error(yg_te, y_gam_pred))
r2   = r2_score(yg_te, y_gam_pred)
corr = np.corrcoef(yg_te, y_gam_pred)[0,1]
biais = np.mean(y_gam_pred - yg_te)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Scatter réel vs prédit
ax1 = axes[0]
c_pts = ['#4CAF50' if v<5 else '#FFC107' if v<10 else '#FF9800' if v<15 else '#F44336'
          for v in yg_te]
ax1.scatter(yg_te, y_gam_pred, c=c_pts, s=15, alpha=0.5)
max_val = max(yg_te.max(), y_gam_pred.max())
ax1.plot([0, max_val],[0, max_val], 'k--', linewidth=2, label='Prédiction parfaite', alpha=0.5)
ax1.set_xlabel('GAM% réel')
ax1.set_ylabel('GAM% prédit')
ax1.set_title(f'GAM Réel vs Prédit\nMAE={mae:.2f}% | R²={r2:.3f}', fontweight='bold')
# Légende seuils OMS
legend_patches = [
    mpatches.Patch(color='#4CAF50', label='Acceptable (<5%)'),
    mpatches.Patch(color='#FFC107', label='Alerte (5-10%)'),
    mpatches.Patch(color='#FF9800', label='Urgence (10-15%)'),
    mpatches.Patch(color='#F44336', label='Crise (≥15%)'),
]
ax1.legend(handles=legend_patches, fontsize=7.5, loc='upper left')
ax1.text(0.05, 0.95, f'r={corr:.3f}', transform=ax1.transAxes,
         fontsize=10, fontweight='bold', va='top')

# Résidus
ax2 = axes[1]
residus = y_gam_pred - yg_te
ax2.scatter(yg_te, residus, c=c_pts, s=12, alpha=0.5)
ax2.axhline(y=0, color='black', linewidth=2)
ax2.axhline(y=biais, color=UNICEF_RED, linestyle='--', linewidth=1.5,
            label=f'Biais moyen = {biais:.2f}%')
ax2.axhspan(-mae, mae, alpha=0.1, color='green', label=f'±MAE ({mae:.2f}%)')
ax2.set_xlabel('GAM% réel')
ax2.set_ylabel('Résidu (prédit - réel)')
ax2.set_title('Graphique des Résidus', fontweight='bold')
ax2.legend(fontsize=8)

# Distribution des erreurs
ax3 = axes[2]
ax3.hist(np.abs(residus), bins=30, color=UNICEF_BLUE, alpha=0.8, edgecolor='white')
ax3.axvline(x=mae, color=UNICEF_RED, linestyle='--', linewidth=2,
            label=f'MAE = {mae:.2f}%')
ax3.axvline(x=rmse, color=UNICEF_ORANGE, linestyle='--', linewidth=2,
            label=f'RMSE = {rmse:.2f}%')
ax3.axvline(x=3.0, color='gray', linestyle=':', linewidth=2, alpha=0.7,
            label='Seuil UNICEF MAE < 3%')
ax3.set_xlabel('Erreur absolue |GAM prédit - GAM réel| (%)')
ax3.set_ylabel('Fréquence')
ax3.set_title('Distribution des Erreurs Absolues', fontweight='bold')
ax3.legend(fontsize=8)

plt.suptitle('Évaluation du Régresseur GAM% — NutritionPredictor',
             fontsize=12, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_gam_regression.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMétriques Régresseur GAM :')
print(f'   MAE  : {mae:.3f}% GAM  → Seuil UNICEF < 3% : {"OK" if mae < 3 else "À améliorer"}')
print(f'   RMSE : {rmse:.3f}%')
print(f'   R²   : {r2:.4f}')
print(f'   Corr : {corr:.4f}')
print(f'   Biais: {biais:.3f}% (sous-estimation si négatif)')
pct_within_3 = np.mean(np.abs(residus) < 3) * 100
print(f'   Précision ±3% : {pct_within_3:.1f}% des prédictions')

## 7. Analyse par Seuils OMS — Performance Critique


In [ ]:
# ── Performance aux seuils humanitaires OMS ─────────────────────
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

seuils_oms = [
    (5,  'Alerte (5%)',    COLOR_ALERTE),
    (10, 'Urgence (10%)', COLOR_URGENCE),
    (15, 'Crise (15%)',   COLOR_CRISE),
]

for i, (seuil, label_seuil, color_seuil) in enumerate(seuils_oms):
    ax = axes[i]

    y_true_seuil = (yg_te >= seuil).astype(int)
    y_pred_seuil = (y_gam_pred >= seuil).astype(int)

    f1   = f1_score(y_true_seuil, y_pred_seuil, zero_division=0)
    rec  = recall_score(y_true_seuil, y_pred_seuil, zero_division=0)
    prec = precision_score(y_true_seuil, y_pred_seuil, zero_division=0)

    # Matrice de confusion
    cm = confusion_matrix(y_true_seuil, y_pred_seuil)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, int(y_true_seuil.sum())

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2%', cmap='Blues',
        ax=ax, cbar=False,
        xticklabels=[f'<{seuil}%', f'≥{seuil}%'],
        yticklabels=[f'<{seuil}%', f'≥{seuil}%'],
        linewidths=0.5, linecolor='gray',
        annot_kws={'size': 11, 'weight': 'bold'},
    )

    ax.set_xlabel('Prédit', fontsize=10)
    ax.set_ylabel('Réel', fontsize=10)
    ax.set_title(
        f'Seuil {label_seuil}\nF1={f1:.3f} | Recall={rec:.3f} | Precision={prec:.3f}',
        fontweight='bold', color=color_seuil
    )
    # Annotation critique : FN (cas urgents manqués)
    ax.text(0.5, -0.18,
            f'Faux Négatifs (cas manqués) : {fn} | Rappel={rec:.3f}',
            transform=ax.transAxes, ha='center', fontsize=8.5,
            color='red' if rec < 0.7 else 'green', fontweight='bold')

plt.suptitle(
    'Performance aux Seuils Humanitaires OMS\n'
    'Focus : Rappel (ne pas manquer les situations critiques)',
    fontsize=12, fontweight='bold', color=UNICEF_DARK
)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_confusion_seuils.png', dpi=150, bbox_inches='tight')
plt.show()

# Synthèse
print('\nSynthèse par seuil OMS (CRITIQUE pour décisions humanitaires) :')
print(f'{"Seuil":15s} {"Recall":10s} {"Précision":10s} {"F1":8s} {"Statut":20s}')
print('-' * 65)
for seuil, label_s, _ in seuils_oms:
    y_t = (yg_te >= seuil).astype(int)
    y_p = (y_gam_pred >= seuil).astype(int)
    rec  = recall_score(y_t, y_p, zero_division=0)
    prec = precision_score(y_t, y_p, zero_division=0)
    f1   = f1_score(y_t, y_p, zero_division=0)
    ok   = 'Acceptable' if rec >= 0.70 else 'À améliorer'
    print(f'{label_s:15s} {rec:10.3f} {prec:10.3f} {f1:8.3f} {ok}')

## 8. Explicabilité SHAP — Ensemble


In [ ]:
# ── SHAP Random Forest (principal contributeur 60%) ──────────────
try:
    import shap

    X_sample = X_te_sc[:150]
    y_clf_sample = (y_te[:150] >= 0.25).astype(int)

    # SHAP TreeExplainer sur RF
    explainer_rf = shap.TreeExplainer(model._rf)
    shap_rf = explainer_rf.shap_values(X_sample)
    # Binary classifier → classe 1
    sv_rf = shap_rf[1] if isinstance(shap_rf, list) else shap_rf

    fig, axes = plt.subplots(1, 2, figsize=(20, 9))

    # Beeswarm summary
    plt.sca(axes[0])
    shap.summary_plot(
        sv_rf, X_sample,
        feature_names=list(feature_names),
        max_display=20,
        show=False,
        color_bar_label='Valeur de la feature (normalisée)',
    )
    axes[0].set_title(
        'SHAP — Impact sur Score de Risque Nutrition\n(Random Forest, 60% de l\'ensemble)',
        fontweight='bold', color=UNICEF_DARK
    )

    # Bar importance SHAP comparée
    plt.sca(axes[1])
    imp_rf  = pd.Series(np.abs(sv_rf).mean(axis=0), index=feature_names).nlargest(15)

    # Catégories couleurs
    cat_colors = {
        'gam_lag': UNICEF_RED,
        'score_fcs': UNICEF_GREEN,
        'score_paludisme': '#9C27B0',
        'prix': UNICEF_ORANGE,
        'soudure': '#795548',
        'indice': UNICEF_DARK,
    }

    def get_color(fname):
        for key, col in cat_colors.items():
            if key in fname:
                return col
        return UNICEF_BLUE

    colors_imp = [get_color(f) for f in imp_rf.index]
    bars = axes[1].barh(imp_rf.index, imp_rf.values,
                         color=colors_imp, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, imp_rf.values):
        axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                     f'{val:.4f}', va='center', fontsize=8)

    # Légende catégories
    legend_cats = [
        mpatches.Patch(color=UNICEF_RED,    label='Historique GAM'),
        mpatches.Patch(color=UNICEF_GREEN,  label='Sécurité alimentaire (FCS)'),
        mpatches.Patch(color='#9C27B0',     label='Feature croisée paludisme'),
        mpatches.Patch(color=UNICEF_ORANGE, label='Prix alimentaires'),
        mpatches.Patch(color=UNICEF_BLUE,   label='Autres features'),
    ]
    axes[1].legend(handles=legend_cats, fontsize=8, loc='lower right')
    axes[1].set_title('Importance SHAP par Feature — Top 15', fontweight='bold')
    axes[1].set_xlabel('Importance SHAP moyenne')

    plt.suptitle('Analyse SHAP — NutritionPredictor\nTransparence Algorithmique UNICEF',
                 fontsize=12, fontweight='bold', color=UNICEF_DARK)
    plt.tight_layout()
    plt.savefig('../../data/processed/nutrition_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nTop 10 features SHAP :')
    for i, (f, v) in enumerate(imp_rf.head(10).items(), 1):
        cat = next((k for k in cat_colors if k in f), 'autre')
        print(f'   {i:2d}. {f:35s}: {v:.4f}  [{cat}]')

except ImportError:
    print('SHAP non disponible — pip install shap')
except Exception as e:
    print(f'SHAP : {e}')

## 9. Saisonnalité — Impact de la Soudure sur la Malnutrition


In [ ]:
# ── Simulation impact saisonnier ─────────────────────────────────
from src.preprocessing.health_processor import HealthProcessor
from src.data_collection.nutrition_fetcher import REGION_FOOD_PROFILE

# Simulation prédiction sur 12 mois pour 3 régions
REGIONS_SIM = ['MDG-ANA', 'MDG_AND', 'MDG-ATS']
REGION_LABELS = {'MDG-ANA': 'Analamanga (Hautes Terres)',
                  'MDG_AND': 'Androy (Grand Sud aride)',
                  'MDG-ATS': 'Atsinanana (Côte Est)'}

import asyncio
from src.preprocessing.feature_engineering import FeatureEngineer

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors_sim = [UNICEF_BLUE, UNICEF_RED, UNICEF_GREEN]

for (rid, label), color in zip(REGION_LABELS.items(), colors_sim):
    scores_mois = []
    gam_mois    = []

    for mois in range(1, 13):
        target_date = date(2024, mois, 15)

        async def get_features(r, d):
            eng = FeatureEngineer()
            return await eng.build_nutrition_features(r, d)

        features = asyncio.run(get_features(rid, target_date))
        pred = model.predict(features, horizon_days=30)
        scores_mois.append(pred['score_risque'])

        gam_p = pred.get('gam_prevu_pct', pred['score_risque'] * 15)
        gam_mois.append(gam_p)

    mois_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']

    axes[0].plot(range(1, 13), scores_mois, marker='o', color=color,
                 linewidth=2.5, markersize=7, label=REGION_LABELS[rid].split('(')[0].strip())
    axes[1].plot(range(1, 13), gam_mois, marker='s', color=color,
                 linewidth=2.5, markersize=7, linestyle='--',
                 label=REGION_LABELS[rid].split('(')[0].strip())

# Marquage soudure typique
for ax in axes:
    ax.axvspan(10.5, 12.5, alpha=0.07, color='orange', label='Pré-soudure')
    ax.axvspan(0.5, 4.5, alpha=0.07, color='orange')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(mois_labels)
    ax.set_xlabel('Mois')

axes[0].set_title('Score de Risque Nutrition par Mois', fontweight='bold')
axes[0].set_ylabel('Score de risque [0, 1]')
axes[0].set_ylim(0, 1)
for seuil, label_s, color_s in [(0.25, 'Risque moyen', 'orange'),
                                   (0.5, 'Risque élevé', 'red')]:
    axes[0].axhline(y=seuil, color=color_s, linestyle=':', alpha=0.5)
axes[0].legend(fontsize=9)

axes[1].set_title('GAM% Prévu par Mois', fontweight='bold')
axes[1].set_ylabel('Taux GAM prévu (%)')
for seuil, col in [(5, COLOR_ALERTE), (10, COLOR_URGENCE), (15, COLOR_CRISE)]:
    axes[1].axhline(y=seuil, color=col, linestyle='--', linewidth=1.5, alpha=0.7,
                    label=f'OMS seuil {seuil}%')
axes[1].legend(fontsize=8, ncol=2)

plt.suptitle('Saisonnalité du Risque Malnutrition — Simulation 2024\n'
             '(Zone grisée = période de soudure typique)',
             fontsize=12, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_saisonnalite.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. RecipeSelector — Recommandations Nutritionnelles


In [ ]:
# ── Test de l'algorithme de sélection de recettes ────────────────
from src.reports.recipe_selector import RecipeSelector

selector = RecipeSelector()

# Test pour différentes régions et cibles
CAS_TESTS = [
    ('MDG_AND',  'enfants_6_23m',    'Androy — Enfants 6-23 mois'),
    ('MDG-ATS',  'femmes_enceintes', 'Atsinanana — Femmes enceintes'),
    ('MDG-ANA',  'famille',          'Analamanga — Famille'),
    ('MDG-BOE',  'enfants_2_5ans',   'Boeny — Enfants 2-5 ans'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for i, (region_id, cible, titre) in enumerate(CAS_TESTS):
    recettes = asyncio.run(
        selector.generer_recettes_optimales(region_id, cible, nombre=4)
    )

    ax = axes[i]

    if recettes:
        noms    = [r['nom'][:40] for r in recettes]
        scores  = [r.get('score_nutritionnel', r.get('_score_optimisation', 60)) for r in recettes]
        proteins = [r.get('proteines_g', 0) for r in recettes]
        fer      = [r.get('fer_mg', 0) for r in recettes]

        x = np.arange(len(noms))
        width = 0.28

        # Normalisation pour affichage
        scores_norm  = [s/100 * 100 for s in scores]
        prot_norm    = [p/20 * 100 for p in proteins]  # /20g ref
        fer_norm     = [f/10 * 100 for f in fer]       # /10mg ref

        b1 = ax.bar(x - width, scores_norm, width, label='Score nutrition (/100)',
                    color=UNICEF_BLUE, alpha=0.85, edgecolor='white')
        b2 = ax.bar(x, prot_norm, width, label='Protéines (% ref 20g)',
                    color=UNICEF_GREEN, alpha=0.85, edgecolor='white')
        b3 = ax.bar(x + width, fer_norm, width, label='Fer (% ref 10mg)',
                    color=UNICEF_RED, alpha=0.85, edgecolor='white')

        ax.set_xticks(x)
        ax.set_xticklabels(
            [n.replace(' ', '\n') if len(n) > 25 else n for n in noms],
            fontsize=7.5, ha='right', rotation=15
        )
        ax.set_ylabel('Score / % référence', fontsize=9)
        ax.set_title(titre, fontweight='bold', fontsize=10)
        ax.axhline(y=70, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        if i == 0:
            ax.legend(fontsize=7.5, loc='upper right')
        ax.set_ylim(0, 140)

        # Annotation coût
        for j, (bar, rec) in enumerate(zip(b1, recettes)):
            cout = rec.get('cout_estime_ariary', rec.get('cout_ariary', 0))
            ax.text(j - width, bar.get_height() + 2,
                    f'{int(cout):,}\nMGA', ha='center', fontsize=6.5, color='navy')

    else:
        ax.text(0.5, 0.5, 'Aucune recette disponible', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='gray')

plt.suptitle('RecipeSelector — Recommandations Nutritionnelles par Région et Cible',
             fontsize=13, fontweight='bold', color=UNICEF_DARK)
plt.tight_layout()
plt.savefig('../../data/processed/nutrition_recettes.png', dpi=150, bbox_inches='tight')
plt.show()

# Évaluation couverture besoins
print('\nCouverture des besoins journaliers — Grand Sud (MDG_AND):')
recettes_and = asyncio.run(
    selector.generer_recettes_optimales('MDG_AND', 'enfants_6_23m', nombre=3)
)
apport = selector.calculer_apport_journalier(recettes_and, repas_par_jour=3)
couverture = selector.evaluer_couverture_besoins(apport, 'enfants_6_23m')

print(f'  Score global de couverture : {couverture["score_global"]}%')
for nutriment, detail in couverture['couverture'].items():
    print(f'  {detail["statut"]} {nutriment:15s}: '
          f'{detail["apport"]:.1f}/{detail["besoin"]:.1f} ({detail["couverture_pct"]}%)')

## 11. Validation Complète et Rapport Final


In [ ]:
# ── Évaluation complète via evaluate.py ─────────────────────────
from ml.training_scripts.evaluate import (
    evaluate_nutrition_model, generate_evaluation_report
)

print('Évaluation complète NutritionPredictor (standards UNICEF)...')
eval_results = evaluate_nutrition_model(
    model=model,
    X_test=X_te_sc,
    y_test=y_te,
    y_gam_test=yg_te,
    feature_names=list(feature_names),
)

# Dashboard de synthèse
fig = plt.figure(figsize=(18, 10))
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.35)

# 1. Jauge score global
ax_gauge = fig.add_subplot(gs[0, 0])
auc_val = eval_results['classification']['auc_roc']
color_auc = COLOR_CRISE if auc_val < 0.70 else COLOR_URGENCE if auc_val < 0.75 \
            else COLOR_ALERTE if auc_val < 0.82 else COLOR_ACCEPTABLE
ax_gauge.pie(
    [auc_val, 1-auc_val],
    colors=[color_auc, '#EEEEEE'],
    startangle=90, counterclock=False,
    wedgeprops=dict(width=0.4, edgecolor='white'),
)
ax_gauge.text(0, 0, f'{auc_val:.3f}', ha='center', va='center',
               fontsize=20, fontweight='bold', color=color_auc)
ax_gauge.set_title('AUC-ROC', fontweight='bold', fontsize=11)

# 2. Métriques classification
ax_clf = fig.add_subplot(gs[0, 1:3])
clf_m = eval_results['classification']
metrics_bar = {
    'AUC-ROC':     (clf_m['auc_roc'],    0.72),
    'F1-Score':    (clf_m['f1_score'],   0.60),
    'Recall':      (clf_m['recall'],     0.68),
    'Précision':   (clf_m['precision'],  0.60),
    'Bal. Acc':    (clf_m['balanced_accuracy'], 0.60),
}
noms_m = list(metrics_bar.keys())
vals_m = [v[0] for v in metrics_bar.values()]
seui_m = [v[1] for v in metrics_bar.values()]
cols_m = [COLOR_ACCEPTABLE if v >= s else COLOR_CRISE
           for v, s in zip(vals_m, seui_m)]
b = ax_clf.barh(noms_m, vals_m, color=cols_m, alpha=0.85, edgecolor='white')
for bar, seuil in zip(b, seui_m):
    ax_clf.axvline(x=seuil, color='gray', linestyle=':', alpha=0.5, linewidth=1)
for bar, val in zip(b, vals_m):
    ax_clf.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
ax_clf.set_xlim(0, 1.15)
ax_clf.set_title('Métriques Classification\n(rouge = sous seuil UNICEF)', fontweight='bold')
ax_clf.set_xlabel('Score')

# 3. Métriques GAM
ax_gam = fig.add_subplot(gs[0, 3])
reg_gam = eval_results.get('regression_gam', {})
gam_metrics = {
    'MAE (%)':  reg_gam.get('mae', 0),
    'RMSE (%)': reg_gam.get('rmse', 0),
    'MAPE (%)': reg_gam.get('mape_pct', 0) / 100,  # Normalisation
}
ax_gam.barh(list(gam_metrics.keys()), list(gam_metrics.values()),
             color=[COLOR_ACCEPTABLE if v <= 3 else COLOR_CRISE
                    for v in [reg_gam.get('mae',0), reg_gam.get('rmse',0), 0]],
             alpha=0.85, edgecolor='white')
ax_gam.axvline(x=3, color='red', linestyle='--', linewidth=1.5,
                label='Seuil MAE < 3%')
ax_gam.set_title('Régresseur GAM\n(seuil MAE < 3%)', fontweight='bold')
ax_gam.legend(fontsize=8)

# 4. Validation UNICEF
ax_valid = fig.add_subplot(gs[1, :2])
valid = eval_results['validation_unicef']
details = valid['details']
noms_v = list(details.keys())
vals_v = [details[k]['valeur'] for k in noms_v]
seui_v = [details[k]['seuil'] for k in noms_v]
ok_v   = [details[k]['statut'].startswith('✅') for k in noms_v]

colors_v = [COLOR_ACCEPTABLE if ok else COLOR_CRISE for ok in ok_v]
b_v = ax_valid.bar(noms_v, vals_v, color=colors_v, alpha=0.85, edgecolor='white')
for bar, seuil, val in zip(b_v, seui_v, vals_v):
    ax_valid.plot([bar.get_x(), bar.get_x() + bar.get_width()], [seuil, seuil],
                   'k--', linewidth=1.5, alpha=0.6)
    ax_valid.text(bar.get_x() + bar.get_width()/2,
                   max(val, seuil) + 0.01,
                   f'{val:.3f}', ha='center', fontsize=8.5, fontweight='bold')
ax_valid.set_title(
    f'Validation Seuils UNICEF — Conformité {valid["score_conformite"]}%',
    fontweight='bold'
)
ax_valid.tick_params(axis='x', rotation=20)

# 5. Calibration
ax_cal = fig.add_subplot(gs[1, 2:])
from sklearn.calibration import calibration_curve
y_clf_te = (y_te >= 0.25).astype(int)
y_pr = model._predict_raw(X_te_sc)
frac, mean_p = calibration_curve(y_clf_te, y_pr, n_bins=8)
ax_cal.plot(mean_p, frac, 'o-', color=UNICEF_BLUE, linewidth=2, markersize=7,
             label=f'NutritionPredictor (Brier={eval_results["calibration"]["brier_score"]:.4f})')
ax_cal.plot([0,1],[0,1], 'k--', alpha=0.4, label='Parfaitement calibré')
ax_cal.fill_between([0,1],[0,0.1],[0.1,0.2], alpha=0.05, color='gray')
ax_cal.set_xlabel('Probabilité prédite')
ax_cal.set_ylabel('Fraction réelle de positifs')
ax_cal.set_title('Courbe de Calibration', fontweight='bold')
ax_cal.legend(fontsize=9)

fig.suptitle(
    'Dashboard d\'Évaluation — NutritionPredictor (Ensemble RF+MLP)\n'
    '  — Standards de transparence algorithmique',
    fontsize=13, fontweight='bold', color=UNICEF_DARK
)
plt.savefig('../../data/processed/nutrition_dashboard_eval.png', dpi=150, bbox_inches='tight')
plt.show()

# Rapport
rapport_path = generate_evaluation_report(
    eval_results,
    Path('../../ml/experiments/nutrition_evaluation_report.json')
)
print(f'\nRapport sauvegardé → {rapport_path}')
print(f'Conformité UNICEF : {valid["score_conformite"]}%')

In [ ]:
# ── Récapitulatif final ──────────────────────────────────────────
print('=' * 75)
print('RÉCAPITULATIF — MODÈLE NUTRITION (ENSEMBLE RF + MLP)')
print('=' * 75)
print(f'  Modèle   : NutritionPredictor v{model.MODEL_VERSION}')
print(f'  Architecture : RandomForest(60%) + MLP(40%) + GBR-GAM')
print(f'  Features : {len(feature_names)} (nutrition, prix, épi, socio-éco, géo)')
print(f'  Feature croisée : score_paludisme (co-morbidité)')

clf_m = eval_results['classification']
gam_m = eval_results.get('regression_gam', {})
valid = eval_results['validation_unicef']

print(f'\n  Métriques finales :')
print(f'    Classification')
print(f'      AUC-ROC     : {clf_m["auc_roc"]:.4f}  (seuil UNICEF ≥ 0.72)')
print(f'      F1-Score    : {clf_m["f1_score"]:.4f}  (seuil UNICEF ≥ 0.60)')
print(f'      Recall      : {clf_m["recall"]:.4f}  (seuil UNICEF ≥ 0.68 — critique)')
print(f'    Régression GAM')
print(f'      MAE         : {gam_m.get("mae",0):.3f}%  (seuil UNICEF < 3%)')
print(f'      R²          : {gam_m.get("r2",0):.4f}')
print(f'      Corrélation : {gam_m.get("correlation",0):.4f}')

print(f'\n  Conformité UNICEF : {valid["score_conformite"]}%')

print(f'\n  Artefacts générés (data/processed/) :')
artefacts = [
    'nutrition_target_dist.png',
    'nutrition_correlations.png',
    'nutrition_comorbidite.png',
    'nutrition_roc_composants.png',
    'nutrition_gam_regression.png',
    'nutrition_confusion_seuils.png',
    'nutrition_shap_summary.png',
    'nutrition_saisonnalite.png',
    'nutrition_recettes.png',
    'nutrition_dashboard_eval.png',
]
for a in artefacts:
    print(f'    ✅ {a}')

print(f'\n  Prochaines étapes :')
print('    1. Entraîner sur données SMART/MICS réelles Madagascar')
print('    2. Intégrer données WFP VAM mensuelles (prix + FCS)')
print('    3. Calibration isotonique pour améliorer Brier Score')
print('    4. Analyse d\'équité régionale (performance Grand Sud)')
print('    5. Déploiement API → /api/v1/nutrition/risque/{region_id}')
print('    6. Pipeline Celery → retraining mensuel automatique')